# Comparing vision-language models with `ammico` and OpenRouter

With `ammico`, you can create image captions (summaries) and answer questions about images using a vision-language model. Which model works best depends on your images and your research question, so it is often useful to run the same analysis with several models and compare the results.

This notebook uses [OpenRouter](https://openrouter.ai), a service that gives access to many hosted models (OpenAI, Google, Anthropic, Qwen, Meta, Mistral, ...) through a single **OpenAI-compatible API** and a single API key. The notebook runs the analysis once per model and saves the results of every model to its own csv file, so that you can compare them side by side.

**Privacy note:** with OpenRouter your images are sent to a third party (OpenRouter and the provider of the selected model). Do not use this setup for data that must not leave your institution.

You can run this notebook on google colab or locally. This first cell only runs on google colab; on all other machines, you need to create a conda environment first and install ammico from the Python Package Index using  
```pip install ammico```  
Alternatively you can install the development version from the GitHub repository  
```pip install git+https://github.com/ssciwr/AMMICO.git```

In [ ]:
# when running on Google colab, otherwise the below cell is skipped
if "google.colab" in str(get_ipython()):
    # uv is a fast Python package manager, see https://github.com/astral-sh/uv
    %pip install uv
    # Install ammico as the latest version from GitHub, which will pull in the compatible dependencies
    !uv pip install git+https://github.com/ssciwr/ammico.git

Now you need to restart the kernel to load the new dependencies. For this, click on "Runtime -> Restart Session" or press Ctrl+M.

<div style="display: flex; justify-content: space-around; align-items: center;">
  <img src="../_static/restart_session.png" alt="Restart Session" style="width: 45%;">
</div>

Now you are ready to import ammico.

In [ ]:
import ammico

# Step 1: Read your data into AMMICO

`ammico` reads in files from a directory. You can iterate through directories in a recursive manner and filter by extensions. Reading in these files creates a dictionary `image_dict`, with one entry per image file, containing the file path and filename. This dictionary is the main data structure that ammico operates on.

For reading in the files, the ammico function `find_files` is used, with optional keywords:

| input key | input type | possible input values |
| --------- | ---------- | --------------------- |
`path` | `str` | the directory containing the image files (defaults to the location set by environment variable `AMMICO_DATA_HOME`) |
| `pattern` | `str\|list` | the file extensions to consider (defaults to "png", "jpg", "jpeg", "gif", "webp", "avif", "tiff") |
| `recursive` | `bool` | include subdirectories recursively (defaults to `True`) |
| `limit` | `int` | maximum number of files to read (defaults to `20`, for all images set to `None` or `-1`) |
| `random_seed` | `int` | the random seed for shuffling the images; applies when only a few images are read and the selection should be preserved (defaults to `None`) |

Since every model is billed per request on OpenRouter, we recommend to start with a small number of images. To compare the models fairly, all models are run on exactly the same images.

In [ ]:
# Define your data path
data_path = "../../data/in"

# Find files and create the image dictionary
image_dict = ammico.find_files(
    path=data_path,
    limit=10,  # Limit the number of files to process (optional)
)

# Step 2: Connect to OpenRouter

To use OpenRouter, you need an account and an API key, which you can create at [openrouter.ai/keys](https://openrouter.ai/keys). The key is read from the environment variable `OPENROUTER_API_KEY`; if it is not set, you are asked to enter it below. Do not write your API key directly into the notebook, in particular if you share the notebook with others!

`ammico` connects to the model via three settings: the base URL, the API key and the model id. For OpenRouter, the base URL and the API key are the same for all models, only the model id changes. Setting `AMMICO_PRIVACY_ACK` to `"True"` acknowledges that the data is sent to an external service and silences the corresponding warning.

In [ ]:
import os
from getpass import getpass

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("Enter your OpenRouter API key: ")

os.environ["AMMICO_API_BASE_URL"] = "https://openrouter.ai/api/v1"
os.environ["AMMICO_API_KEY"] = os.environ["OPENROUTER_API_KEY"]
os.environ["AMMICO_PRIVACY_ACK"] = "True"  # acknowledge sending data externally

# Step 3: Select the models to compare

Any model on OpenRouter that accepts images as input can be used. You can find them on the [OpenRouter model list](https://openrouter.ai/models?input_modalities=image), where you also see the price per token. The model id is the name shown on the model page, prefixed with the provider, for example `openai/gpt-4o-mini` or `google/gemini-2.5-flash`.

Add or remove model ids in the list below. The results of each model are saved to its own csv file in `results_dir`. Models for which a result file already exists are skipped, so you can add models to the list later and re-run the notebook without paying again for the models you already ran. Images that failed for a model (for example because the provider was temporarily rate-limited) are not lost: when you re-run the notebook, only these images are sent to the model again and the results are merged into the existing file. Set `overwrite = True` to re-run all models on all images.

In [ ]:
models = [
    "openai/gpt-4o-mini",
    "google/gemini-2.5-flash",
    "qwen/qwen2.5-vl-72b-instruct",
    "meta-llama/llama-4-maverick",
]

results_dir = "./openrouter_results"  # one csv file per model is written here
overwrite = False  # set to True to re-run models that already have a result file

Some models "think" before they answer (reasoning models, for example the Qwen3 or Gemini 2.5 models). The reasoning counts towards the token limit of the answer, so that the caption may come back empty. The class below asks OpenRouter to switch off the reasoning for all models; set `disable_reasoning = False` if you want to keep it. Note that a few models cannot switch off their reasoning; these may then return an error or an empty caption.

Many models on OpenRouter are served from a pool that is shared between all users of a provider. When this pool is busy, the request is rejected with a **rate limit error** (`Error code: 429`, "temporarily rate-limited upstream"). The class below therefore waits and retries such requests (and requests that failed because the provider was temporarily unavailable), with a waiting time that doubles after each attempt. Only if the request still fails after `max_retries` attempts, the image is skipped. To avoid running into the rate limit in the first place, the number of requests sent at the same time is limited by `max_parallel_images` (images processed at the same time) and `max_parallel_requests` (requests per image, one per question). Lower these numbers if you still see many rate limit errors.

In [ ]:
import random
import time

import openai

disable_reasoning = True
max_retries = (
    6  # retries of a rate-limited request; waits about 5, 10, 20, 40, 80, 120 s
)
max_parallel_images = 4  # number of images processed at the same time
max_parallel_requests = 2  # number of requests (questions) per image at the same time


class OpenRouterModel(ammico.InferenceModel):
    """InferenceModel that retries rate-limited requests and optionally switches off reasoning."""

    def chat(self, messages, max_new_tokens=256, n=1):
        extra_body = {"reasoning": {"enabled": False}} if disable_reasoning else None
        for attempt in range(max_retries + 1):
            try:
                response = self.client.chat.completions.create(
                    model=self.model_id,
                    messages=messages,
                    max_tokens=max_new_tokens,
                    n=n,
                    temperature=0,
                    extra_body=extra_body,
                )
                break
            except (
                openai.RateLimitError,  # 429: rate-limited
                openai.InternalServerError,  # 5xx: provider temporarily unavailable
                openai.APIConnectionError,
                openai.APITimeoutError,
            ) as e:
                if attempt == max_retries:
                    raise  # give up, the image is skipped with a warning
                wait = min(5 * 2**attempt, 120) + random.uniform(0, 2)
                print(
                    f"{self.model_id}: {type(e).__name__}, retry {attempt + 1}/{max_retries} in {wait:.0f} s"
                )
                time.sleep(wait)
        return [(choice.message.content or "").strip() for choice in response.choices]

# Step 4: Run the analysis for all models

Now we define what we want to know about the images. You can choose between creating a caption (`"summary"`), answering your questions (`"questions"`) or both (`"summary_and_questions"`). With `is_concise_summary` and `is_concise_answer` you control whether the model answers briefly or in more detail. See the [image summary tutorial](https://github.com/ssciwr/AMMICO/blob/main/docs/tutorials/image_summary.md) for more details on the options.

In [ ]:
analysis_type = (
    "summary_and_questions"  # "summary", "questions" or "summary_and_questions"
)
list_of_questions = [
    "Who is in the picture?",
    "Is Trump in the picture, answer with only yes or no?",
]  # add or replace with your own questions
is_concise_summary = True
is_concise_answer = True

The cell below loops over all selected models. For each model, the analysis is run on a fresh copy of `image_dict`, so that the results of the different models do not overwrite each other. The results are converted into a dataframe and saved as `<results_dir>/<provider>__<model>.csv`; the `/` in the model id is replaced since it is not allowed in file names. The results of all models are also collected in the dictionary `results`.

Depending on the number of images and models, this can take several minutes. If a request still fails after all retries (for example because of a persistent rate limit, a misspelled model id or a model that does not accept images), a warning is printed and the image is skipped, leaving an empty result. The images with results are saved nevertheless. When you execute the cell again, the images that already have results are skipped and only the missing images are sent to the model again; their results are merged into the existing file. So if a model was rate-limited, simply wait a bit and run the cell again. If a model does not return any results at all, nothing is saved for this model; the loop then continues with the next model.

In [ ]:
import copy
from pathlib import Path

import pandas as pd

Path(results_dir).mkdir(parents=True, exist_ok=True)

# the result columns that must be filled for an image to count as done
result_columns = [
    column
    for column, kind in [("caption", "summary"), ("vqa", "questions")]
    if kind in analysis_type
]


def result_file(model_id):
    """The csv file that holds the results of a model."""
    return Path(results_dir) / f"{model_id.replace('/', '__')}.csv"


def is_missing(value):
    """Failed requests leave an empty caption or an empty list of answers."""
    if isinstance(value, (list, tuple)):  # answers before saving to csv
        return len(value) == 0
    return pd.isna(value) or str(value).strip() in ("", "[]")


def failed_images(df):
    """Boolean mask of the rows of df that have no (complete) result."""
    if any(column not in df.columns for column in result_columns):
        return pd.Series(True, index=df.index)
    return df[result_columns].map(is_missing).any(axis=1)


results = {}
for model_id in models:
    outfile = result_file(model_id)
    previous = None
    todo = image_dict
    if outfile.exists() and not overwrite:
        previous = pd.read_csv(outfile, index_col=0)
        done = set(previous.loc[~failed_images(previous), "filename"].astype(str))
        todo = {
            key: entry
            for key, entry in image_dict.items()
            if str(entry["filename"]) not in done
        }
        if not todo:
            print(f"{model_id}: skipped, all results already in {outfile}")
            continue
        print(
            f"{model_id}: {len(todo)} images without results, running these again ..."
        )
    else:
        print(f"{model_id}: running on {len(todo)} images ...")

    start = time.time()
    try:
        model = OpenRouterModel(
            model_id=model_id, max_concurrency=max_parallel_requests
        )
        model_dict = copy.deepcopy(todo)  # keep the results of each model separate
        image_summary_vqa = ammico.ImageSummaryDetector(
            summary_model=model, subdict=model_dict
        )
        model_dict = image_summary_vqa.analyse_images_from_dict(
            analysis_type=analysis_type,
            list_of_questions=list_of_questions,
            is_concise_summary=is_concise_summary,
            is_concise_answer=is_concise_answer,
            keys_batch_size=max_parallel_images,
        )
    except Exception as e:
        print(f"{model_id}: failed with {type(e).__name__}: {e}")
        continue

    df = ammico.get_dataframe(model_dict)
    df.insert(0, "model", model_id)
    n_new = int((~failed_images(df)).sum())
    if n_new == 0:
        print(
            f"{model_id}: no results returned, check the warnings above - nothing saved"
        )
        continue

    if previous is not None:
        # replace the old (failed) rows of the re-run images by the new results
        previous = previous[
            ~previous["filename"].astype(str).isin(df["filename"].astype(str))
        ]
        df = pd.concat([previous, df], ignore_index=True)
    df.to_csv(outfile)
    results[model_id] = df

    n_missing = int(failed_images(df).sum())
    print(
        f"{model_id}: {n_new} new results in {time.time() - start:.1f} s, saved to {outfile}"
    )
    if n_missing:
        print(
            f"{model_id}: {n_missing} images still without results - run this cell again later to retry them"
        )

# Step 5: Compare the results

The cell below reads the result files of all selected models back in. This way, the comparison also includes the models that were skipped above because they had been run before, for example in an earlier session.

In [ ]:
all_results = {
    model_id: pd.read_csv(result_file(model_id), index_col=0)
    for model_id in models
    if result_file(model_id).exists()
}
print(f"Results available for {len(all_results)} of {len(models)} models.")

To compare the captions, we put them into one table with one row per image and one column per model:

In [ ]:
column = "caption"  # use "vqa" to compare the answers to the questions instead

comparison = pd.DataFrame(
    {model_id: df.set_index("filename")[column] for model_id, df in all_results.items()}
)
pd.set_option("display.max_colwidth", None)
comparison

The comparison table can be saved to a csv file as well:

In [ ]:
comparison.to_csv(Path(results_dir) / f"comparison_{column}.csv")

For a closer look, we display each image together with the captions of all models:

In [ ]:
import textwrap

import matplotlib.pyplot as plt
from PIL import Image

for image_path, captions in comparison.iterrows():
    img = Image.open(image_path)
    plt.figure(figsize=(8, 6))
    plt.imshow(img)
    plt.axis("off")  # Hide axes
    plt.tight_layout()
    plt.show()
    for model_id, caption in captions.items():
        print(f"{model_id}:")
        print(textwrap.indent(textwrap.fill(str(caption), width=100), "    "))
    print()

If you want to keep all results in a single file instead, for example to analyze them further in another program, you can also combine the dataframes of all models into one long table. Each row then corresponds to one image analyzed by one model, and the column `model` tells you which model produced the result.

In [ ]:
combined = pd.concat(all_results.values(), ignore_index=True)
combined.to_csv(Path(results_dir) / "all_models.csv")
combined.head(5)

# Step 6: Check out further notebooks or create your own!

Congratulations! You have compared several vision-language models with `ammico`. Check out [the documentation](https://github.com/ssciwr/AMMICO) for further tutorials on how to extract text from images or analyze video content! Do not hesitate to [get in touch](https://github.com/ssciwr/AMMICO/issues) with questions, feedback or any technical issues!